In [20]:
###RAG PIPELINES - DATA INGESTION TO VECTOR DB PIPELINE

In [21]:
import os
from langchain_community.document_loaders import PyPDFLoader,PyMuPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from pathlib import Path

In [22]:
### Read all the pdf's inside the directory pdf_files

from pathlib import Path
from langchain_community.document_loaders import PyPDFLoader

def process_all_pdfs(pdf_directory):
    all_documents = []
    pdf_dir = Path(pdf_directory)

    # Find all PDFs recursively
    pdf_files = list(pdf_dir.glob("**/*.pdf"))

    print(f"Found {len(pdf_files)} PDF files to process")

    for pdf_file in pdf_files:
        print(f"Processing: {pdf_file.name}")
        try:
            loader = PyPDFLoader(str(pdf_file))
            documents = loader.load()

            # Add source information to metadata
            for doc in documents:
                doc.metadata['source_file'] = pdf_file.name
                doc.metadata['file_type'] = 'pdf'

            # Extend OUTSIDE the loop
            all_documents.extend(documents)

        except Exception as e:
            print(f"X ERROR: {e}")

    print(f"\nTotal documents loaded: {len(all_documents)}")
    return all_documents




In [23]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

def split_document(documents, chunk_size=1000, chunk_overlap=200):
    """Split documents into smaller chunks for better RAG performance"""

    text_splitter = RecursiveCharacterTextSplitter(
        chunk_size=chunk_size,
        chunk_overlap=chunk_overlap,
        length_function=len,
        separators=["\n\n", "\n", " ", ""]
    )

    split_docs = text_splitter.split_documents(documents)

    print(f"Split {len(documents)} documents into {len(split_docs)} chunks")

    # Show example of a chunk
    if split_docs:
        print("\nExample chunk:")
        print(f"Content: {split_docs[0].page_content[:200]}...")
        print(f"Metadata: {split_docs[0].metadata}")

    return split_docs

In [28]:
documents = process_all_pdfs("../data/pdf_files")
chunks = split_document(documents)
print(f"Documents loaded: {len(documents)}")
print(f"Chunks created: {len(chunks)}")

Found 2 PDF files to process
Processing: PythonFinance.pdf
Processing: PythonProgramming.pdf

Total documents loaded: 854
Split 854 documents into 1790 chunks

Example chunk:
Content: Yves Hilpisch
Python  
for Finance
MASTERING DATA-DRIVEN FINANCE
2nd Edition...
Metadata: {'producer': 'Antenna House PDF Output Library 2.6.0 (Linux64)', 'creator': 'AH CSS Formatter V6.0 MR2 for Linux64 : 6.0.2.5372 (2012/05/16 18:26JST)', 'creationdate': '2020-07-27T13:22:30+00:00', 'author': 'Yves Hilpisch', 'ebx_publisher': "O'Reilly Media, Incorporated", 'moddate': '2021-05-23T12:45:31+03:00', 'title': 'Python for Finance', 'trapped': '/False', 'source': '..\\data\\pdf_files\\PythonFinance.pdf', 'total_pages': 714, 'page': 0, 'page_label': 'Cover', 'source_file': 'PythonFinance.pdf', 'file_type': 'pdf'}
Documents loaded: 854
Chunks created: 1790


### embedding and vectorstoreDB


In [ ]:
import numpy as np
from sentence_transformers import SentenceTransformer
import chromadb
from chromadb.config import Settings
import uuid
from typing import List, Dict, Any, Tuple
from sklearn.metrics.pairwise import cosine_similarity

### class for embedding

In [29]:
import numpy as np
from sentence_transformers import SentenceTransformer
from typing import List, Union


class EmbeddingManager:
    """
    Handles document embedding generation using Sentence Transformers
    """

    def __init__(self, model_name: str = "all-MiniLM-L6-v2"):
        """
        Initialize the embedding manager

        Args:
            model_name: HuggingFace model name for sentence embeddings
        """
        self.model_name = model_name
        self.model = None
        self._load_model()

    def _load_model(self):
        """Load the SentenceTransformer model"""
        try:
            print(f"Loading embedding model: {self.model_name}")

            self.model = SentenceTransformer(self.model_name)

            print(
                f"Model loaded successfully. Embedding dimension: "
                f"{self.model.get_sentence_embedding_dimension()}"
            )

        except Exception as e:
            print(f"Error loading model {self.model_name}: {e}")
            raise

    def generate_embeddings(self, texts: Union[str, List[str]]) -> np.ndarray:
        """
        Generate embeddings for input texts

        Args:
            texts: A string or list of text strings

        Returns:
            numpy array of embeddings with shape (num_texts, embedding_dim)
        """

        if self.model is None:
            raise ValueError("Model not loaded")

        # Convert single string to list
        if isinstance(texts, str):
            texts = [texts]

        print(f"Generating embeddings for {len(texts)} texts...")

        embeddings = self.model.encode(
            texts,
            show_progress_bar=True,
            convert_to_numpy=True
        )

        print(f"Generated embeddings with shape: {embeddings.shape}")

        return embeddings

    def get_embedding_dimension(self) -> int:
        """Return embedding dimension of the model"""

        if self.model is None:
            raise ValueError("Model not loaded")

        return self.model.get_sentence_embedding_dimension()


# Initialize the embedding manager
embedding_manager = EmbeddingManager()
embedding_manager

Loading embedding model: all-MiniLM-L6-v2


Loading weights: 100%|██████████| 103/103 [00:00<00:00, 8077.13it/s]
BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Model loaded successfully. Embedding dimension: 384


### vectorStore


In [30]:
import os
import uuid
import chromadb
import numpy as np
from typing import List, Any


class VectorStore:
    """Manages document embeddings in a ChromaDB vector store"""

    def __init__(self, collection_name: str = "pdf_documents",
                 persist_directory: str = "../data/vector_store"):

        """
        Initialize the vector store

        Args:
            collection_name: Name of the ChromaDB collection
            persist_directory: Directory to persist the vector store
        """

        self.collection_name = collection_name
        self.persist_directory = persist_directory
        self.client = None
        self.collection = None

        self._initialize_store()

    def _initialize_store(self):
        """Initialize ChromaDB client and collection"""

        try:
            # Create directory if not exists
            os.makedirs(self.persist_directory, exist_ok=True)

            # Create persistent client
            self.client = chromadb.PersistentClient(path=self.persist_directory)

            # Create / get collection
            self.collection = self.client.get_or_create_collection(
                name=self.collection_name,
                metadata={"description": "PDF document embeddings for RAG"}
            )

            print(f"Vector store initialized. Collection: {self.collection_name}")
            print(f"Existing documents: {self.collection.count()}")

        except Exception as e:
            print(f"Error initializing vector store: {e}")
            raise

    def add_documents(self, documents: List[Any], embeddings: np.ndarray):
        """
        Add documents and their embeddings to the vector store

        Args:
            documents: List of LangChain documents
            embeddings: Corresponding embeddings
        """

        if len(documents) != len(embeddings):
            raise ValueError("Number of documents must match number of embeddings")

        print(f"Adding {len(documents)} documents to vector store...")

        ids = []
        metadatas = []
        documents_text = []
        embeddings_list = []

        for i, (doc, embedding) in enumerate(zip(documents, embeddings)):

            # Generate unique ID
            doc_id = f"doc_{uuid.uuid4().hex[:8]}_{i}"
            ids.append(doc_id)

            # Metadata
            metadata = dict(doc.metadata)
            metadata["doc_index"] = i
            metadata["content_length"] = len(doc.page_content)
            metadatas.append(metadata)

            # Document text
            documents_text.append(doc.page_content)

            # Embedding
            embeddings_list.append(embedding.tolist())

        try:
            self.collection.add(
                ids=ids,
                embeddings=embeddings_list,
                metadatas=metadatas,
                documents=documents_text
            )

            print(f"Successfully added {len(documents)} documents")
            print(f"Total documents in collection: {self.collection.count()}")

        except Exception as e:
            print(f"Error adding documents: {e}")
            raise


# Initialize vector store
vector_store = VectorStore()

Vector store initialized. Collection: pdf_documents
Existing documents: 12508


In [ ]:
chunks

[Document(metadata={'producer': 'Antenna House PDF Output Library 2.6.0 (Linux64)', 'creator': 'AH CSS Formatter V6.0 MR2 for Linux64 : 6.0.2.5372 (2012/05/16 18:26JST)', 'creationdate': '2020-07-27T13:22:30+00:00', 'author': 'Yves Hilpisch', 'ebx_publisher': "O'Reilly Media, Incorporated", 'moddate': '2021-05-23T12:45:31+03:00', 'title': 'Python for Finance', 'trapped': '/False', 'source': '..\\data\\pdf_files\\PythonFinance.pdf', 'total_pages': 714, 'page': 0, 'page_label': 'Cover', 'source_file': 'PythonFinance.pdf', 'file_type': 'pdf'}, page_content='Yves Hilpisch\nPython  \nfor Finance\nMASTERING DATA-DRIVEN FINANCE\n2nd Edition'),
 Document(metadata={'producer': 'Antenna House PDF Output Library 2.6.0 (Linux64)', 'creator': 'AH CSS Formatter V6.0 MR2 for Linux64 : 6.0.2.5372 (2012/05/16 18:26JST)', 'creationdate': '2020-07-27T13:22:30+00:00', 'author': 'Yves Hilpisch', 'ebx_publisher': "O'Reilly Media, Incorporated", 'moddate': '2021-05-23T12:45:31+03:00', 'title': 'Python for Fi

In [31]:
texts=[doc.page_content for doc in chunks]
texts

['Yves Hilpisch\nPython  \nfor Finance\nMASTERING DATA-DRIVEN FINANCE\n2nd Edition',
 'Yves Hilpisch\nPython for Finance\nMastering Data-Driven Finance\nSECOND EDITION\nBoston Farnham Sebastopol TokyoBeijing Boston Farnham Sebastopol TokyoBeijing',
 '978-1-492-02433-0\n[MBP]\nPython for Finance\nby Yves Hilpisch\nCopyright © 2019 Yves Hilpisch. All rights reserved.\nPrinted in Canada.\nPublished by O’Reilly Media, Inc., 1005 Gravenstein Highway North, Sebastopol, CA 95472.\nO’Reilly books may be purchased for educational, business, or sales promotional use. Online editions are\nalso available for most titles (http://oreilly.com). For more information, contact our corporate/institu‐\ntional sales department: 800-998-9938 or corporate@oreilly.com.\nEditors: \nSusan Conant and Jeff Bleiel Indexer: Judith McConville\nProduction Editor: Kristen Brown Interior Designer: David Futato\nCopyeditor: Rachel Head Cover Designer: Karen Montgomery\nProofreader: Kim Cofer Illustrator: Rebecca Demares

In [ ]:
valid_docs_final = []
embeddings = []

# iterate over the chunks returned by split_document
for doc in chunks:
    try:
        text = doc.page_content

        emb = embedding_manager.generate_embeddings(text)

        embeddings.append(emb[0])   # because encode returns (1, embedding_dim)
        valid_docs_final.append(doc)

    except Exception as e:
        # log the exception so we know which chunk failed
        print("Skipped a problematic chunk:", e)


Generating embeddings for 1 texts...


Batches:   0%|          | 0/1 [00:00<?, ?it/s]

Batches: 100%|██████████| 1/1 [00:00<00:00, 19.15it/s]


Generated embeddings with shape: (1, 384)
Generating embeddings for 1 texts...


Batches: 100%|██████████| 1/1 [00:00<00:00, 30.58it/s]


Generated embeddings with shape: (1, 384)
Generating embeddings for 1 texts...


... (1,145 outputs left out)

Generated embeddings with shape: (1, 384)
Generating embeddings for 1 texts...


Batches: 100%|██████████| 1/1 [00:00<00:00,  6.22it/s]


Generated embeddings with shape: (1, 384)
Generating embeddings for 1 texts...


Batches: 100%|██████████| 1/1 [00:00<00:00,  3.28it/s]

In [ ]:
import numpy as np

embeddings = np.array(embeddings)

## stored in vector db


In [ ]:
vector_store.add_documents(valid_docs_final, embeddings)

Adding 1787 documents to vector store...
Successfully added 1787 documents
Total documents in collection: 12508


## skipped the broken chunks 


In [ ]:
print("Original valid chunks:", len(chunks))
print("Embeddings created:", len(valid_docs_final))
print("Skipped:", len(chunks) - len(valid_docs_final))


Original valid chunks: 1790
Embeddings created: 1787
Skipped: 3


## data retrieval


In [ ]:
from google import genai
print("Gemini SDK loaded successfully")

Gemini SDK loaded successfully


In [ ]:
class Retriever:
    """Handles query embedding and retrieval from vector store"""

    def __init__(self, embedding_manager, vector_store):
        self.embedding_manager = embedding_manager
        self.vector_store = vector_store

    def retrieve(self, query, k=3):
        query_embedding = self.embedding_manager.generate_embeddings(query)

        results = self.vector_store.collection.query(
            query_embeddings=query_embedding.tolist(),
            n_results=k
        )

        return results["documents"][0]

In [25]:
retriever = Retriever(embedding_manager, vector_store)

In [26]:
from dotenv import load_dotenv
import os
from google import genai

# Load environment variables
load_dotenv()

# Initialize Gemini client
client = genai.Client(api_key=os.getenv("GEMINI_API_KEY"))


def answer_question(query, k=10):
    """
    Retrieve relevant document chunks and generate answer using Gemini
    """

    # Step 1 — retrieve relevant chunks
    docs = retriever.retrieve(query, k)

    if not docs:
        return "No relevant information found in the document."

    # Step 2 — limit chunk size to avoid huge prompts
    context = "\n\n".join(doc[:500] for doc in docs)

    # Step 3 — build the prompt
    prompt = f"""
You are a helpful assistant answering questions based on the provided document context.

Context:
{context}

Question:
{query}

Answer the question using ONLY the context above.
If the answer is not present, say: "The document does not contain this information."
"""

    # Step 4 — call Gemini API
    response = client.models.generate_content(
        model="gemini-2.5-flash",
        contents=prompt
    )

    return response.text




In [ ]:
query = "What is Python?"

answer = answer_question(query)

print(answer)

Generating embeddings for 1 texts...


Batches: 100%|██████████| 1/1 [00:00<00:00, 27.33it/s]

Generated embeddings with shape: (1, 384)


ValueError: Expected requested number of results to be a int, got <__main__.Retriever object at 0x00000264E1435650> in query.